# Privacy-preserving federated learning: serial C vs CUDA
**Reference article:** Kaissis, Makowski, Rueckert, Braren (2020), *Secure, privacy-preserving and federated machine learning in medical imaging*, Nature Machine Intelligence 2, 305-311. [doi:10.1038/s42256-020-0186-1](https://doi.org/10.1038/s42256-020-0186-1)

**The idea in one paragraph:** hospitals want to train one model together without sharing patient images. Two privacy tricks make that safe, but both cost extra computation. We time those two tricks on a normal CPU (serial C) and on a GPU (CUDA) to see how much the GPU helps.

| Kernel | What it does (plain words) | Implementations compared |
|---|---|---|
| **DP gradient** | clips each image's gradient and adds noise, so no single patient can be recognised (differential privacy) | serial C, CUDA naive, CUDA optimised |
| **Secure aggregation** | each hospital hides its update behind random masks that only cancel out in the sum | serial C, CUDA |

**How to run:** run the dependency cell, fill in the team details, then **Run all** (Colab: *Runtime > Change runtime type > GPU* first for the CUDA part; without a GPU everything else still runs).
The last cell writes `report/report.pdf`.

## Dependencies (run this first)

In [ ]:
# Python packages used by the notebook
%pip install -q numpy pandas matplotlib reportlab

# System tools (already present on Colab; gcc is required, the NVIDIA tools only for the CUDA part)
import shutil
for tool in ("gcc", "nvcc", "nvidia-smi"):
    print(f"{tool:11s}", "found" if shutil.which(tool) else ("NOT found - required" if tool == "gcc" else "NOT found - CUDA part will be skipped"))

## 0. Team details and settings (fill in, then Run all)

In [ ]:
# ===================== TEAM DETAILS (appear on the PDF; blanks print as "[to be filled]") =====================
PROJECT_TITLE = "Parallel Acceleration of Privacy-Preserving Federated Learning for Medical Imaging"
COURSE        = ""        # e.g. "CS-xxx High Performance Computing, Prof. ..."
INSTITUTION   = ""

# One entry per member. Edit the names/IDs; the roles are a suggested split, change them if yours differ.
MEMBERS = [
    dict(name="", id="",
         role="Member 1 - GPU lead: CUDA kernels (naive + optimised) and the benchmarks",
         contribution="",    # what you did, 2-5 sentences
         reflection=""),     # what you learned
    dict(name="", id="",
         role="Member 2 - Privacy and data lead: dataset, serial C baseline, DP / secure aggregation, federated training",
         contribution="",
         reflection=""),
    dict(name="", id="",
         role="Member 3 - Results lead: correctness checks, figures, report and presentation",
         contribution="",
         reflection=""),
]
# Fewer than 3 people? Delete the extra entries (and merge the roles). More people? Copy an entry.

# Your own interpretation of the results (a paragraph). Printed in Section 9 of the report.
OBSERVATIONS = ""

# ===================== RUN SETTINGS =====================
QUICK = False   # True: smaller sizes, finishes in ~1 minute (good for a first test)

## 1. Setup (locate the project folder)

In [ ]:
import os, sys, glob, zipfile, shutil, subprocess, time, math, statistics, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def find_root():
    for p in (os.getcwd(), os.path.dirname(os.getcwd())):
        if os.path.exists(os.path.join(p, "src", "fedkernels.py")):
            return p
    return None

ROOT = find_root()
if ROOT is None:                                   # e.g. fresh Colab session: unzip the project
    zips = glob.glob("fedprivacy-cuda*.zip")
    if not zips:
        try:
            from google.colab import files
            print("Please upload fedprivacy-cuda.zip")
            zips = list(files.upload())
        except ImportError:
            raise SystemExit("Put this notebook inside the project folder (next to src/ and data/).")
    zipfile.ZipFile(zips[0]).extractall(".")
    ROOT = os.path.abspath("fedprivacy-cuda")
os.chdir(ROOT); sys.path.insert(0, os.path.join(ROOT, "src"))

from fedkernels import Kernels, system_info, dp_grad_numpy, secagg_reference, dp_epsilon_zcdp, SCALE

FIG = os.path.join(ROOT, "report", "figs"); RES = os.path.join(ROOT, "results")
os.makedirs(FIG, exist_ok=True); os.makedirs(RES, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "font.size": 10})
REPS = 2 if QUICK else 3
CLIP = 5.0                                         # DP clipping norm C
rng = np.random.default_rng(0)
print("Project root:", ROOT)

## 2. Hardware check and kernel build (gcc, plus nvcc + CUDA if a GPU exists)

In [ ]:
K = Kernels().build()
assert K.cpu, "gcc is required (Linux, Colab or WSL). On macOS install gcc via Homebrew."
SYS = system_info(); SYS["gpu"] = f"{K.gpu_name} ({K.gpu_mem_gb:.1f} GB)" if K.cuda else None
HAS_CUDA = K.cuda is not None
DP_BACKENDS = K.backends_dp          # serial, and cuda_naive / cuda_opt if a GPU exists
SA_BACKENDS = K.backends_secagg      # serial, and cuda if a GPU exists
print("DP gradient backends:", DP_BACKENDS, "| secure aggregation backends:", SA_BACKENDS)
pd.Series(SYS).to_frame("value")

## 3. Dataset: 5 simulated hospitals (non-IID synthetic chest X-rays)

In [ ]:
npz_path = os.path.join(ROOT, "data", "synthetic_chest_xray_32.npz")
if not os.path.exists(npz_path):
    subprocess.run([sys.executable, os.path.join(ROOT, "scripts", "make_dataset.py")], check=True)
d = np.load(npz_path)
IMG, Y, HOSP, SPLIT = d["X"], d["y"].astype(np.float32), d["hospital"], d["split"]
S = IMG.shape[1]

def featurise(img):                                # per-image standardisation (a purely LOCAL step, no cross-hospital statistics)
    x = img.reshape(len(img), -1).astype(np.float32)
    x = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-6) * 0.25   # ||x|| ~ 8
    x = np.hstack([x, np.ones((len(x), 1), np.float32)])               # bias term
    pad = (-x.shape[1]) % 32
    return np.hstack([x, np.zeros((len(x), pad), np.float32)])

FEAT = featurise(IMG); D = FEAT.shape[1]
tr, te = SPLIT == 0, SPLIT == 1
Xtr_all, ytr_all = FEAT[tr], Y[tr]
Xte, yte = FEAT[te], Y[te]
clients = []                                       # per-hospital training data (never pooled during FL)
for h in sorted(set(HOSP.tolist())):
    m = (HOSP == h) & tr
    X_, y_, xn_ = Kernels.prep(FEAT[m], Y[m])
    clients.append(dict(h=h, X=X_, y=y_, xn=xn_, n=int(m.sum())))
ds_rows = [[str(h), str(int((HOSP == h).sum())), f"{Y[HOSP == h].mean():.3f}", str(int(((HOSP == h) & te).sum()))]
           for h in sorted(set(HOSP.tolist()))]
DATASET = dict(n_total=len(Y), size=S, D=D, rows=ds_rows)

fig, ax = plt.subplots(2, 8, figsize=(12, 3.4))
for r, lab in enumerate((0, 1)):
    for c, i in enumerate(np.where(Y == lab)[0][:8]):
        ax[r, c].imshow(IMG[i], cmap="gray", vmin=0, vmax=255); ax[r, c].axis("off")
fig.suptitle("Synthetic chest-X-ray-like images (top row: normal, bottom row: pneumonia-like)")
fig.tight_layout(); fig.savefig(f"{FIG}/fig_samples.png", dpi=150); plt.show()
print(f"{len(Y)} images, feature length D = {D}"); pd.DataFrame(ds_rows, columns=["hospital", "images", "pneumonia rate", "test images"])

## 4. Correctness: do all implementations agree?
Every fast version is compared against a simple NumPy reference. If the numbers match, the speed-ups below are trustworthy.

In [ ]:
Nc = 4000
Xc, yc, xnc = Kernels.prep(Xtr_all[:Nc], ytr_all[:Nc])
wc = rng.normal(0, 0.05, D).astype(np.float32)
ref = dp_grad_numpy(Xc, yc, wc, CLIP)
if HAS_CUDA: K.upload(Xc, yc, xnc)
correct_rows = []
for b in DP_BACKENDS:
    g, _ = K.dp_grad(b, Xc, yc, xnc, wc, CLIP)
    err = float(np.abs(g - ref).max() / np.abs(ref).max())
    ok_ = err < 1e-3
    correct_rows.append([b, f"{err:.2e}" + ("" if ok_ else "  (CHECK!)")])
    if not ok_: print(f"WARNING: {b} deviates from the reference by {err:.2e}")
print(pd.DataFrame(correct_rows, columns=["backend", "max rel. error vs float64 numpy"]).to_string(index=False))

# secure aggregation: masks cancel, and one client's masked share looks random
Kc, Dc = 5, 200_000
Uc = rng.normal(0, 0.1, (Kc, Dc)).astype(np.float32)
sref = secagg_reference(Uc); bound = Kc * 0.5 / SCALE
outs = {b: K.secagg(b, Uc)[0] for b in SA_BACKENDS}
for b, o in outs.items():
    e = float(np.abs(o - sref).max())
    print(f"secagg {b:7s}: max abs error {e:.2e}  (fixed-point bound {bound:.2e})" + ("" if e <= bound * 1.01 else "   WARNING: above bound"))
if HAS_CUDA:
    same = np.array_equal(outs["serial"], outs["cuda"])
    print("CPU and CUDA secure-aggregation outputs bit-identical:", same)
share = K.secagg_share(Uc, 0).astype(np.float64)
corr = float(np.corrcoef(share, Uc[0])[0, 1])
SECAGG_CHECK = (f"Check: the server-side sum agrees with the float64 sum to within {max(float(np.abs(o - sref).max()) for o in outs.values()):.1e} "
                f"(fixed-point bound {bound:.1e}). A single client's masked share has correlation {corr:+.3f} with its true update, "
                f"i.e. it reveals nothing about it on its own.")
print(SECAGG_CHECK)

## 5. Benchmark A - DP gradient kernel vs batch size (serial / CUDA naive / CUDA optimised)

In [ ]:
ram = SYS.get("ram_gb") or 8
N_LIST = [2_000, 10_000, 50_000] if QUICK else [2_000, 10_000, 50_000, 100_000, 200_000]
N_LIST = [n for n in N_LIST if n * D * 4 <= 0.25 * ram * 2**30]      # keep within ~25% of RAM
Nmax = max(N_LIST)
idx = np.arange(Nmax) % len(Xtr_all)                                 # tile the real data (timing is data-independent)
Xb, yb, xnb = Kernels.prep(Xtr_all[idx], ytr_all[idx])
wb = rng.normal(0, 0.05, D).astype(np.float32)

rows, upload_ms = [], {}
for N in N_LIST:
    X_, y_, xn_ = Xb[:N], yb[:N], xnb[:N]
    if HAS_CUDA: upload_ms[N] = min(K.upload(X_, y_, xn_) for _ in range(3)) * 1e3
    for b in DP_BACKENDS:
        K.dp_grad(b, X_, y_, xn_, wb, CLIP)                          # warm-up
        ts, ks = [], []
        for _ in range(REPS):
            _, t = K.dp_grad(b, X_, y_, xn_, wb, CLIP); ts.append(t)
            if b.startswith("cuda"): ks.append(K.last_kernel_s)
        rows.append(dict(N=N, backend=b, ms=statistics.median(ts) * 1e3,
                         kernel_ms=statistics.median(ks) * 1e3 if ks else np.nan))
    print(f"N={N:>7}: " + "  ".join(f"{r['backend']}={r['ms']:.2f}ms" for r in rows if r["N"] == N))
dp = pd.DataFrame(rows)
dp["gflops"] = 4 * dp.N * D / (dp.ms / 1e3) / 1e9
dp["gbs"] = 2 * dp.N * D * 4 / (dp.ms / 1e3) / 1e9
dp["speedup"] = dp.apply(lambda r: dp[(dp.N == r.N) & (dp.backend == "serial")].ms.iloc[0] / r.ms, axis=1)
dp.round(3).to_csv(f"{RES}/benchmark_dp_gradient.csv", index=False)
if HAS_CUDA: K.cuda.gpu_free()
dp.pivot(index="N", columns="backend", values="ms").round(2)

## 6. Benchmark B - CUDA *with* and *without* transfer cost (needs a GPU)
Copying data to the GPU is not free. "Per step" assumes the hospital data already sits on the GPU; "upload + 1 step" is the worst case.

In [ ]:
xf = None
if HAS_CUDA:
    xr = []
    for N in N_LIST:
        o = dp[(dp.N == N) & (dp.backend == "cuda_opt")].iloc[0]
        ser = dp[(dp.N == N) & (dp.backend == "serial")].iloc[0].ms
        up = upload_ms[N]
        gain = ser - o.ms
        xr.append(dict(N=N, upload_ms=up, kernel_ms=o.kernel_ms, step_ms=o.ms, oneshot_ms=up + o.ms, serial_ms=ser,
                       breakeven=(up / gain if gain > 0 else float("inf"))))
    xf = pd.DataFrame(xr); xf.round(3).to_csv(f"{RES}/benchmark_cuda_transfer.csv", index=False)
    display(xf.round(2))
else:
    print("No GPU in this run -> CUDA sections are skipped. In Colab: Runtime > Change runtime type > T4 GPU, then Run all.")

## 7. Benchmark C - secure aggregation kernel (K clients x D parameters)

In [ ]:
KC = 10
D_LIST = [100_000, 1_000_000, 5_000_000] if QUICK else [100_000, 1_000_000, 5_000_000, 10_000_000]
D_LIST = [x for x in D_LIST if KC * x * 4 <= 0.2 * ram * 2**30]
s_rows = []
for Dm in D_LIST:
    U = rng.normal(0, 0.1, (KC, Dm)).astype(np.float32)
    for b in SA_BACKENDS:
        K.secagg(b, U)                                               # warm-up
        res = [K.secagg(b, U, seed=7) for _ in range(REPS)]
        s_rows.append(dict(D=Dm, backend=b, ms=statistics.median(r[1] for r in res) * 1e3,
                           kernel_ms=statistics.median(r[2] for r in res) * 1e3))
    del U
sa = pd.DataFrame(s_rows)
sa["speedup"] = sa.apply(lambda r: sa[(sa.D == r.D) & (sa.backend == "serial")].ms.iloc[0] / r.ms, axis=1)
sa.round(3).to_csv(f"{RES}/benchmark_secagg.csv", index=False)
sa.pivot(index="D", columns="backend", values="ms").round(2)

## 8. End-to-end: federated training with DP + secure aggregation
Each hospital trains locally with clipped, noised gradients (DP-GD). Only the weight updates leave the hospital, and they are combined with the secure-aggregation kernel. More noise (larger sigma) means stronger privacy (smaller epsilon) but lower accuracy.

In [ ]:
ROUNDS, LOCAL, LR, DELTA = 30, 4, 0.1, 1e-5
TRAIN_BACKEND = "serial"
SEC_BACKEND = "cuda" if HAS_CUDA else "serial"
ntot = sum(c["n"] for c in clients)

def accuracy(w):
    return float((((Xte @ w) > 0).astype(np.float32) == yte).mean())

def fed_train(sigma, clip, seed=1):
    r = np.random.default_rng(seed); w = np.zeros(D, np.float32); curve = []
    for rd in range(ROUNDS):
        ups = []
        for c in clients:
            wl = w.copy()
            for _ in range(LOCAL):
                g, _ = K.dp_grad(TRAIN_BACKEND, c["X"], c["y"], c["xn"], wl, clip)
                g = g + r.normal(0, sigma * clip, D).astype(np.float32)          # Gaussian mechanism
                wl = wl - LR * g / c["n"]
            ups.append((c["n"] / ntot) * (wl - w))                              # weighted update
        agg, _, _ = K.secagg(SEC_BACKEND, np.ascontiguousarray(np.stack(ups).astype(np.float32)), seed=rd)
        w = w + agg; curve.append(accuracy(w))
    return curve

settings = [("No privacy (no clip, no noise)", 0.0, 1e6), ("DP sigma=1", 1.0, CLIP), ("DP sigma=3", 3.0, CLIP), ("DP sigma=6", 6.0, CLIP)]
runs = {}
for name, sg, cl in settings:
    runs[name] = dict(sigma=sg, curve=fed_train(sg, cl), eps=dp_epsilon_zcdp(sg, ROUNDS * LOCAL, DELTA) if cl < 1e5 else float("inf"))
    print(f"{name:32s} final acc {runs[name]['curve'][-1]:.3f}   epsilon {runs[name]['eps']:.1f}")
train_rows = [[n, f"{r['sigma']:g}", "inf" if math.isinf(r["eps"]) else f"{r['eps']:.1f}", f"{r['curve'][-1]:.3f}", f"{max(r['curve']):.3f}"]
              for n, r in runs.items()]
pd.DataFrame(train_rows, columns=["setting", "sigma", "epsilon", "final acc", "best acc"])

## 9. Figures

In [ ]:
COL = {"serial": "#555555", "cuda_naive": "#ff7f0e", "cuda_opt": "#2ca02c", "cuda": "#2ca02c"}
LAB = {"serial": "Serial C", "cuda_naive": "CUDA naive", "cuda_opt": "CUDA optimised", "cuda": "CUDA end-to-end"}
figs = {"fig_samples": f"{FIG}/fig_samples.png"}

fig, ax = plt.subplots(figsize=(6.2, 4))
for b in DP_BACKENDS:
    s = dp[dp.backend == b]; ax.loglog(s.N, s.ms, "o-", color=COL[b], label=LAB[b])
ax.set_xlabel("batch size N (rows, D=%d)" % D); ax.set_ylabel("time per DP-gradient call (ms)"); ax.legend(); ax.set_title("DP gradient kernel: time")
fig.tight_layout(); fig.savefig(f"{FIG}/fig_dp_time.png", dpi=150); figs["fig_dp_time"] = f"{FIG}/fig_dp_time.png"; plt.show()

fig, ax = plt.subplots(figsize=(6.2, 4))
for b in DP_BACKENDS:
    s = dp[dp.backend == b]; ax.semilogx(s.N, s.speedup, "o-", color=COL[b], label=LAB[b])
ax.axhline(1, color="k", lw=.8); ax.set_xlabel("batch size N"); ax.set_ylabel("speed-up vs serial"); ax.legend(); ax.set_title("DP gradient kernel: speed-up")
fig.tight_layout(); fig.savefig(f"{FIG}/fig_dp_speedup.png", dpi=150); figs["fig_dp_speedup"] = f"{FIG}/fig_dp_speedup.png"; plt.show()

if HAS_CUDA:
    fig, ax = plt.subplots(figsize=(6.2, 4)); x = np.arange(len(xf)); wd = .27
    ax.bar(x - wd, xf.kernel_ms, wd, label="kernel only", color="#2ca02c")
    ax.bar(x, xf.step_ms, wd, label="per step (+w/g copies)", color="#98df8a")
    ax.bar(x + wd, xf.oneshot_ms, wd, label="upload + 1 step", color="#d62728")
    ax.plot(x, xf.serial_ms, "ko--", label="Serial C"); ax.set_yscale("log")
    ax.set_xticks(x); ax.set_xticklabels(xf.N); ax.set_xlabel("batch size N"); ax.set_ylabel("ms (log)"); ax.legend(fontsize=8)
    ax.set_title("CUDA optimised: with vs without transfer"); fig.tight_layout()
    fig.savefig(f"{FIG}/fig_xfer.png", dpi=150); figs["fig_xfer"] = f"{FIG}/fig_xfer.png"; plt.show()

fig, ax = plt.subplots(figsize=(6.2, 4))
for b in SA_BACKENDS:
    s = sa[sa.backend == b]; ax.loglog(s.D, s.ms, "o-", color=COL[b], label=LAB[b])
if HAS_CUDA:
    s = sa[sa.backend == "cuda"]; ax.loglog(s.D, s.kernel_ms, "^--", color="#2ca02c", label="CUDA kernel only")
ax.set_xlabel(f"model parameters D (K={KC} clients)"); ax.set_ylabel("ms"); ax.legend(); ax.set_title("Secure aggregation")
fig.tight_layout(); fig.savefig(f"{FIG}/fig_secagg.png", dpi=150); figs["fig_secagg"] = f"{FIG}/fig_secagg.png"; plt.show()

fig, ax = plt.subplots(figsize=(6.2, 4))
for n, r in runs.items(): ax.plot(range(1, ROUNDS + 1), r["curve"], label=n + ("" if math.isinf(r["eps"]) else f" (eps={r['eps']:.0f})"))
ax.set_xlabel("federated round"); ax.set_ylabel("test accuracy"); ax.legend(fontsize=8); ax.set_title("Federated training under DP")
fig.tight_layout(); fig.savefig(f"{FIG}/fig_train.png", dpi=150); figs["fig_train"] = f"{FIG}/fig_train.png"; plt.show()

## 10. Automatic findings (computed from the measurements above)

In [ ]:
def at(N, b): return dp[(dp.N == N) & (dp.backend == b)].iloc[0]
Nl = max(N_LIST); F = []
ser = at(Nl, "serial")
bests = dp[dp.N == Nl].sort_values("ms").iloc[0]
if HAS_CUDA:
    n_, c_ = at(Nl, "cuda_naive"), at(Nl, "cuda_opt")
    F.append(f"At the largest batch (N={Nl}) serial C needs {ser.ms:.1f} ms per DP-gradient call; CUDA optimised needs {c_.ms:.2f} ms, "
             f"which is {c_.speedup:.1f}x faster.")
    F.append(f"CUDA optimised is {n_.ms / c_.ms:.1f}x faster than CUDA naive ({c_.ms:.2f} vs {n_.ms:.2f} ms): "
             f"how the kernel reads memory matters as much as using a GPU at all. It reaches {c_.gbs:.0f} GB/s effective bandwidth ({c_.gflops:.0f} GFLOP/s).")
    xl = xf[xf.N == Nl].iloc[0]
    F.append(f"Transfer cost: uploading the N={Nl} batch takes {xl.upload_ms:.0f} ms, so a single call with a fresh copy "
             f"({xl.oneshot_ms:.0f} ms) is {'slower' if xl.oneshot_ms > xl.serial_ms else 'still faster'} than serial C ({xl.serial_ms:.0f} ms); "
             + (f"the upload is paid back after about {xl.breakeven:.1f} calls with resident data." if np.isfinite(xl.breakeven) else "serial C is never beaten."))
    cross = [int(r.N) for r in xf.itertuples() if r.step_ms < r.serial_ms]
    F.append("The GPU (data already resident) beats serial C from N=" + (str(min(cross)) if cross else "- (never in the tested range)") + " upwards.")
else:
    F.append(f"No GPU was available in this run, so only the serial C baseline was measured ({ser.ms:.1f} ms at N={Nl}). Re-run on a GPU for the CUDA comparison.")
sl = sa[sa.D == sa.D.max()]
bs = sl.sort_values("ms").iloc[0]
F.append(f"Secure aggregation of {KC} x {int(sl.D.iloc[0]):,} parameters: " + (f"best end-to-end backend is {LAB[bs.backend]} ({bs.ms:.1f} ms, {bs.speedup:.1f}x over serial)." if bs.speedup >= 1.25 else f"no backend was meaningfully faster than serial ({bs.ms:.0f} ms best).")
         + (f" CUDA kernel-only is {sl[sl.backend == 'cuda'].kernel_ms.iloc[0]:.2f} ms, so copying the updates to the GPU dominates ({sl[sl.backend == 'cuda'].ms.iloc[0]:.1f} ms)." if HAS_CUDA else ""))
nb_, db_ = list(runs.values())[0], list(runs.values())[-1]
F.append(f"Privacy/accuracy trade-off: test accuracy {nb_['curve'][-1]:.3f} without privacy vs {db_['curve'][-1]:.3f} with sigma={db_['sigma']:g} "
         f"(epsilon about {db_['eps']:.1f} at delta={DELTA:g}, conservative zCDP bound).")
for f in F: print("-", f)

## 11. Build the report (PDF)

In [ ]:
from xml.sax.saxutils import escape
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib.utils import ImageReader
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, KeepTogether

def f2(x): return f"{x:.2f}" if x < 100 else f"{x:.0f}"
BLANK = "[to be filled]"
PDF_PATH = os.path.join(ROOT, "report", "report.pdf")

st = getSampleStyleSheet()
H1 = ParagraphStyle("H1", parent=st["Heading2"], spaceBefore=10, spaceAfter=4)
BODY = ParagraphStyle("B", parent=st["BodyText"], fontSize=9.5, leading=13)
CELL = ParagraphStyle("C", parent=st["BodyText"], fontSize=8, leading=10)
def P(t, s=BODY): return Paragraph(escape(str(t)) if t else BLANK, s)

def table(header, rows, widths=None):
    data = [[Paragraph(f"<b>{escape(str(h))}</b>", CELL) for h in header]] + [[Paragraph(escape(str(c)), CELL) for c in r] for r in rows]
    t = Table(data, colWidths=widths, repeatRows=1)
    t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#e8e8e8")), ("GRID", (0, 0), (-1, -1), .4, colors.grey),
                           ("VALIGN", (0, 0), (-1, -1), "TOP")]))
    return t

def figure(key, caption, width=11.5 * cm):
    if key not in figs: return []
    w, h = ImageReader(figs[key]).getSize()
    return [KeepTogether([Image(figs[key], width=width, height=width * h / w), Paragraph(f"<i>{escape(caption)}</i>", CELL)]), Spacer(1, 6)]

BK = DP_BACKENDS
piv = dp.pivot(index="N", columns="backend", values="ms")
eff = [[LAB[b], f2(at(Nl, b).ms), "-" if not b.startswith("cuda") else f2(at(Nl, b).kernel_ms), f"{at(Nl, b).speedup:.1f}x",
        f"{at(Nl, b).gflops:.1f}", f"{at(Nl, b).gbs:.1f}"] for b in BK]
sp = sa.pivot(index="D", columns="backend", values="ms")
sk = sa[sa.backend == "cuda"].set_index("D").kernel_ms if HAS_CUDA else None

best_txt = (f"The fastest DP-gradient implementation at N={Nl} is {LAB[bests.backend]} ({bests.speedup:.1f}x over serial C)."
            if bests.speedup >= 1.25 else f"On this machine no backend beat the serial baseline at N={Nl}.")
concl = best_txt + " " + F[-1] + (" The GPU therefore makes the privacy overhead of federated medical-image learning small, provided data stays on the GPU and memory is accessed carefully."
                                  if bests.speedup >= 2 else " Repeat the run on a GPU machine before drawing conclusions.")

S = []
S += [Paragraph(escape(PROJECT_TITLE), st["Title"]), Paragraph(escape(" - ".join(x for x in (COURSE, INSTITUTION) if x) or "Course project report"), st["Normal"]), Spacer(1, 8)]

S += [Paragraph("Team and roles", H1),
      table(["Name", "ID", "Role"], [[m["name"] or BLANK, m["id"] or BLANK, m["role"] or BLANK] for m in MEMBERS], [4 * cm, 3 * cm, 10 * cm])]

S += [Paragraph("1. What this project does", H1),
      P("Hospitals can train a shared model without sending patient images to each other (federated learning). Two privacy techniques protect the updates: "
        "differential privacy (clip each image's gradient and add noise) and secure aggregation (hide each hospital's update behind random masks that cancel in the sum). "
        "Both add computation. We implement them in serial C and in CUDA and measure how much the GPU saves. Reference: Kaissis et al., Nature Machine Intelligence 2, 305-311 (2020)."),
      Paragraph("2. Setup", H1),
      table(["Item", "Value"], [[k, v if v else "none"] for k, v in SYS.items()], [4 * cm, 13 * cm]),
      Spacer(1, 4), P(f"Each timing is the median of {REPS} runs after a warm-up run."),
      Paragraph("3. Dataset", H1),
      P(f"{DATASET['n_total']} synthetic chest-X-ray-like images ({DATASET['size']}x{DATASET['size']}), split across 5 simulated hospitals with different pneumonia rates (non-IID). Feature length D = {DATASET['D']}."),
      table(["Hospital", "Images", "Pneumonia rate", "Test images"], DATASET["rows"])]
S += figure("fig_samples", "Sample images (top: normal, bottom: pneumonia-like).", 15 * cm)

S += [Paragraph("4. Correctness", H1), P("Each implementation is compared with a float64 NumPy reference."),
      table(["Backend", "Max relative error"], correct_rows), Spacer(1, 4), P(SECAGG_CHECK)]

S += [Paragraph("5. DP gradient kernel", H1), P("Time per call in milliseconds (lower is better)."),
      table(["N"] + [LAB[b] for b in BK], [[f"{n:,}"] + [f2(piv.loc[n, b]) for b in BK] for n in piv.index]), Spacer(1, 6)]
S += figure("fig_dp_time", "Time per DP-gradient call.") + figure("fig_dp_speedup", "Speed-up relative to serial C.")
S += [P(f"Efficiency at the largest batch (N={Nl:,}):"), Spacer(1, 3),
      table(["Backend", "Time (ms)", "Kernel-only (ms)", "Speed-up", "GFLOP/s", "GB/s"], eff)]

if HAS_CUDA:
    S += [Paragraph("6. CUDA with and without transfer cost", H1),
          P("Kernel-only is the GPU compute time. Per step adds the small copies of the weights and gradient. Upload + 1 step is the worst case with fresh data. "
            "Calls to amortise is how many calls with resident data it takes to pay back the one-time upload."),
          table(["N", "Upload (ms)", "Kernel-only (ms)", "Per step (ms)", "Upload+1 step (ms)", "Serial (ms)", "Calls to amortise"],
                [[f"{r.N:,}", f2(r.upload_ms), f2(r.kernel_ms), f2(r.step_ms), f2(r.oneshot_ms), f2(r.serial_ms),
                  f"{r.breakeven:.1f}" if np.isfinite(r.breakeven) else "never"] for r in xf.itertuples()]), Spacer(1, 6)]
    S += figure("fig_xfer", "CUDA optimised, with vs without transfer.")

S += [Paragraph("7. Secure aggregation", H1), P(f"Time in milliseconds to combine {KC} clients' updates (lower is better)."),
      table(["D (params)"] + [LAB[b] for b in SA_BACKENDS] + (["CUDA kernel-only"] if HAS_CUDA else []),
            [[f"{Dm:,}"] + [f2(sp.loc[Dm, b]) for b in SA_BACKENDS] + ([f2(sk.loc[Dm])] if HAS_CUDA else []) for Dm in sp.index]), Spacer(1, 6)]
S += figure("fig_secagg", "Secure aggregation time vs model size.")

S += [Paragraph("8. Federated training under privacy", H1),
      P(f"{ROUNDS} rounds, {LOCAL} local steps per round, clipping norm {CLIP:g}, delta = {DELTA:g}. DP gradient on {TRAIN_BACKEND}, secure aggregation on {SEC_BACKEND}. "
        "Larger sigma means more noise: smaller epsilon (stronger privacy) but lower accuracy."),
      table(["Setting", "sigma", "epsilon", "Final acc.", "Best acc."], train_rows), Spacer(1, 6)]
S += figure("fig_train", "Test accuracy per round.")

S += [Paragraph("9. Findings and observations", H1)] + [P("- " + f) for f in F] + [Spacer(1, 4), P(OBSERVATIONS)]
S += [Paragraph("10. Conclusion", H1), P(concl)]
S += [Paragraph("11. Individual contributions", H1)]
for m in MEMBERS:
    S += [P(f"{m['name'] or BLANK} ({m['id'] or BLANK}) - {m['role'] or BLANK}", ParagraphStyle("M", parent=BODY, fontName="Helvetica-Bold")),
          P("Contribution: " + (m["contribution"] or BLANK)), P("Reflection: " + (m["reflection"] or BLANK)), Spacer(1, 4)]

SimpleDocTemplate(PDF_PATH, pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm, topMargin=2 * cm, bottomMargin=2 * cm,
                  title=PROJECT_TITLE).build(S)
print("Created:", PDF_PATH)
try:
    from google.colab import files
    files.download(PDF_PATH)
except ImportError:
    pass

## 12. Done - submit and commit
* **Submit:** `report/report.pdf` (figures: `report/figs/`, raw numbers: `results/*.csv`).
* **Git:** `git add . && git commit -m "Federated DP kernels: serial vs CUDA" && git push` (see README).
* Re-run the whole notebook on a GPU machine (Colab T4) before submitting so the CUDA sections are filled in.